### ASPO PDF Extractor

#### Overview

Downloads and extracts examination regulation content from the official 
Hof University ASPO PDF, then filters sections by student relevance into 
a structured JSON format for the RAG knowledge base.

- **Source URL:** `https://www.hof-university.com/fileadmin/user_upload/studienbuero/spo-s/spo-in-englisch/ASPO_2024-01_EN.pdf`
- **Tools:** `requests`, `PyPDF2`, `re`
- **Output:** `aspo_priority_sections.json`

---

#### Pipeline Steps

| Step | Script | Description |
|------|--------|-------------|
| 1 | PDF Download | Downloads ASPO PDF from Hof University website |
| 2 | PDF Parsing | Extracts text and splits into sections using §, Part, and Chapter patterns |
| 3 | Priority Filtering | Filters sections into high and medium priority groups |

---

#### Priority Sections

| Priority | Count | Examples |
|----------|-------|---------|
| High | 13 | Exam registration, grading, repetition, thesis requirements |
| Medium | 11 | Deadline extensions, group exams, compensation for disadvantages |
| Excluded | — | Administrative procedures, document retention, examiner qualifications |

In [ ]:
import requests
import os

# List of PDF URLs
pdf_urls = [
    "https://www.hof-university.com/fileadmin/user_upload/studienbuero/spo-s/spo-in-englisch/ASPO_2024-01_EN.pdf",
 
]

# Create folder
os.makedirs("data", exist_ok=True)
os.makedirs("debug_data", exist_ok=True)

for url in pdf_urls:
    filename = os.path.join("debug_data", url.split("/")[-1])
    
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed: {url}")

Downloaded: debug_data/ASPO_2024-01_EN.pdf


In [5]:
import PyPDF2
import json
import re
import warnings
import sys
import os

# Suppress all warnings
warnings.filterwarnings('ignore')

def extract_pdf_to_json_simple(pdf_path, output_json_path):
    """
    Extract PDF content with section titles as keys and details as values.
    Only includes sections that have details.
    """
    
    # Suppress stderr completely during PDF reading
    original_stderr = sys.stderr
    sys.stderr = open(os.devnull, 'w')
    
    try:
        # Open the PDF
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Patterns to identify section headers
            part_pattern = re.compile(r'^(Part\s+\d+)$')
            section_pattern = re.compile(r'^(§\s*\d+)$')
            chapter_pattern = re.compile(r'^(Chapter\s+\d+)$')
            
            # Extract text from all pages
            full_text = ""
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                full_text += page.extract_text()
    finally:
        # Restore stderr
        sys.stderr.close()
        sys.stderr = original_stderr
    
    # Split into lines
    lines = full_text.split('\n')
    
    current_section = None
    sections_list = []
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # Skip empty lines
        if not line:
            i += 1
            continue
        
        # Check if this is a header (Part, Chapter, or §)
        is_header = (part_pattern.match(line) or 
                    section_pattern.match(line) or 
                    chapter_pattern.match(line))
        
        if is_header:
            # Save previous section if exists
            if current_section:
                sections_list.append(current_section)
            
            # Get the title (next line)
            title = ""
            if i + 1 < len(lines):
                title = lines[i + 1].strip()
            
            # Create new section
            current_section = {
                "header": line,
                "title": title,
                "details": ""
            }
            
            i += 2  # Skip header and title
            continue
        
        # Add content to current section
        if current_section is not None:
            if current_section["details"]:
                current_section["details"] += " " + line
            else:
                current_section["details"] = line
        
        i += 1
    
    # Add the last section
    if current_section:
        sections_list.append(current_section)
    
    # Clean up details and restructure
    restructured_data = {}
    
    for section in sections_list:
        # Clean up details - remove extra spaces
        details = re.sub(r'\s+', ' ', section["details"]).strip()
        
        # Only include sections with details
        if details:
            # Use title as key, details as value
            restructured_data[section["title"]] = details
    
    # Write to JSON file
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(restructured_data, f, indent=2, ensure_ascii=False)
    
    print(f"Extraction complete! JSON saved to {output_json_path}")
    print(f"Extracted {len(restructured_data)} sections with details")
    
    return restructured_data


# Usage
pdf_path = "debug_data/ASPO_2024-01_EN.pdf"  # Replace with your PDF path
output_json_path = "debug_data/ASPO_2024-01_EN.json"

os.makedirs("data", exist_ok=True)


result = extract_pdf_to_json_simple(pdf_path, output_json_path)

Extraction complete! JSON saved to debug_data/ASPO_2024-01_EN.json
Extracted 55 sections with details


## Take important sections only

In [6]:
import json
from typing import Dict

class ASPOCleaner:
    """Simple ASPO data extractor - minimal processing"""
    
    def __init__(self, json_file_path: str):
        """Initialize with ASPO JSON file"""
        with open(json_file_path, 'r', encoding='utf-8') as f:
            self.aspo_data = json.load(f)
        
        # Reference link to append
        self.reference_link = (
            "For more information, please visit: "
            "https://www.hof-university.com/studying-at-hof-university/services-and-support/"
            "student-affairs/study-and-examination-regulations.html"
        )
        
        # Define priority sections
        self.high_priority = [
            'Standard Dates, Deadlines',
            'Registration for Examinations',
            'Grading  of Individual  Examinations',
            'Repetition of Examinations',
            'Written Examinations',
            'Oral Examinations',
            'Presentations',
            'Final Theses',
            'Study Paper',
            'Take Home Exam',
            'Project Work',
            'Portfolio Examination',
            'Final Examination, Modules, Credit Points'
        ]
        
        self.medium_priority = [
            'General Admission Requirements',
            'Credit Transfer for Academic  Achievements and  Examination Results',
            'Pre-conditions  for Examination',
            'Group Examinations',
            'Extensions of Deadlines',
            'Compensation for Disadvantages',
            'Dishonesty',
            'Violations  of Order',
            'Dual Studies',
            'Retention  of Examination Documents',
            'Examiners, Aids',
        ]
    
    def extract_priority_sections(self) -> Dict[str, str]:
        """Extract high and medium priority sections and append reference link"""
        extracted = {
            'high_priority': {},
            'medium_priority': {}
        }
        
        # Extract high priority
        for section_name in self.high_priority:
            if section_name in self.aspo_data:
                text = self.aspo_data[section_name]
                text += "\n\n" + self.reference_link
                extracted['high_priority'][section_name] = text
        
        # Extract medium priority
        for section_name in self.medium_priority:
            if section_name in self.aspo_data:
                text = self.aspo_data[section_name]
                text += "\n\n" + self.reference_link
                extracted['medium_priority'][section_name] = text
        
        return extracted
    
    def save_extracted_data(self, output_file: str = 'data/aspo_priority_sections.json'):
        """Save extracted sections to JSON file"""
        extracted = self.extract_priority_sections()
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(extracted, f, indent=2, ensure_ascii=False)
        
        print(f"\n Extracted {len(extracted['high_priority'])} high priority sections")
        print(f" Extracted {len(extracted['medium_priority'])} medium priority sections")
        print(f" Saved to: {output_file}")
        
        return extracted


def main():
    """Main execution"""
    cleaner = ASPOCleaner('debug_data/ASPO_2024-01_EN.json')
    extracted = cleaner.save_extracted_data()
    
    # Print what was extracted
    print("\n  High Priority Sections:")
    for section in extracted['high_priority'].keys():
        print(f"   • {section}")
    
    print("\n  Medium Priority Sections:")
    for section in extracted['medium_priority'].keys():
        print(f"   • {section}")


if __name__ == "__main__":
    main()



 Extracted 13 high priority sections
 Extracted 11 medium priority sections
 Saved to: data/aspo_priority_sections.json

  High Priority Sections:
   • Standard Dates, Deadlines
   • Registration for Examinations
   • Grading  of Individual  Examinations
   • Repetition of Examinations
   • Written Examinations
   • Oral Examinations
   • Presentations
   • Final Theses
   • Study Paper
   • Take Home Exam
   • Project Work
   • Portfolio Examination
   • Final Examination, Modules, Credit Points

  Medium Priority Sections:
   • General Admission Requirements
   • Credit Transfer for Academic  Achievements and  Examination Results
   • Pre-conditions  for Examination
   • Group Examinations
   • Extensions of Deadlines
   • Compensation for Disadvantages
   • Dishonesty
   • Violations  of Order
   • Dual Studies
   • Retention  of Examination Documents
   • Examiners, Aids
